# Week 11 · Day 5 — English → Urdu Translation (Seq2Seq)

The capstone of the sequence-models week: an **Encoder–Decoder** that reads an English sentence and writes the Urdu translation. This is where the two GRUs have **different jobs** — one *understands*, one *generates*.

```
 English sentence  →  Encoder GRU  →  context state  →  Decoder GRU  →  Urdu sentence
```

**Today:**
1. The big idea: encoder–decoder
2. Prepare the English–Urdu data
3. Build the seq2seq model (with **teacher forcing**)
4. Train and **translate** new sentences
5. **Swap the embedding**: learned → **Word2Vec** → **GloVe**, and compare

> **Kaggle GPU:** Settings → Accelerator → GPU. Add the **english-corpus.txt** and **urdu-corpus.txt** files as inputs.
> **Internet ON** for the Word2Vec/GloVe downloads in Part 5.

---
## 1. The big idea: Encoder–Decoder

Translation is **many-to-many**, but input and output lengths differ ("I am a student" = 4 words → "میں ایک طالب علم ہوں" = 5 words). We use **two** GRUs:

- **Encoder GRU** — reads the whole English sentence and squeezes it into a single **context vector** (its final hidden state). Its job: *understand*.
- **Decoder GRU** — starts from that context and generates the Urdu translation **one word at a time**. Its job: *generate*.

```
 "I" → "am" → "a" → "student"          (encoder reads English)
                        ↓
                  context state
                        ↓
 <sos> → میں → طالب → علم → ہوں → <eos>   (decoder writes Urdu)
```

The **`<sos>`** (start) and **`<eos>`** (end) tokens tell the decoder when to begin and stop.

> 🖼️ **IMAGE NEEDED** — search prompt: **"encoder decoder seq2seq GRU machine translation architecture diagram"**  
> *(The standard encoder–decoder diagram: encoder RNN → context vector → decoder RNN producing output words. Central visual for the whole day.)*

In [2]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import pad_sequences
from tensorflow.keras.layers import Input, Embedding, GRU, Dense
from tensorflow.keras.models import Model

tf.random.set_seed(42)
print("TF:", tf.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"))

TF: 2.18.1 | GPUs: []


---
## 2. Prepare the data

We have a **parallel corpus**: two files with aligned lines — line *n* of the English file translates to line *n* of the Urdu file.

In [3]:
# set these to your Kaggle input paths (check the file browser on the right)
EN_PATH = r"D:\AI\Artificial-Intelligence-Machine-Learning-and-Deep-Learning-Corvit-Peshawar-2026\Artificial-Inteligence-Machine-Learning-and-Deep-Learning-NAVTTC-Course-2026\week-11-sequence-models-and-NLP\datasets\raw\english-corpus.txt"   # e.g. /kaggle/input/eng-urdu/english-corpus.txt
UR_PATH = r"D:\AI\Artificial-Intelligence-Machine-Learning-and-Deep-Learning-Corvit-Peshawar-2026\Artificial-Inteligence-Machine-Learning-and-Deep-Learning-NAVTTC-Course-2026\week-11-sequence-models-and-NLP\datasets\raw\urdu-corpus.txt"

en_lines = open(EN_PATH, encoding="utf-8").read().strip().split("\n")
ur_lines = open(UR_PATH, encoding="utf-8").read().strip().split("\n")

# keep only non-empty, aligned pairs
pairs = [(e.strip(), u.strip()) for e, u in zip(en_lines, ur_lines) if e.strip() and u.strip()]
print("total pairs:", len(pairs))
for e, u in pairs[:5]:
    print(f"  {e:30s} -> {u}")

total pairs: 24524
  is zain your nephew            -> زین تمہارا بھتیجا ہے۔
  i wish youd trust me           -> کاش تم مجھ پر بھروسہ کرتے
  did he touch you               -> کیا اس نے آپ کو چھوا؟
  its part of life               -> اس کی زندگی کا حصہ
  zain isnt ugly                 -> زین بدصورت نہیں ہے۔


In [4]:
# to train fast in class, use a subset of the short sentences
MAX_PAIRS = 8000
pairs = [(e, u) for e, u in pairs if len(e.split()) <= 6 and len(u.split()) <= 8][:MAX_PAIRS]

en_texts = [e for e, u in pairs]
# wrap every Urdu target with start/end tokens so the decoder knows where to begin & stop
ur_texts = ["<sos> " + u + " <eos>" for e, u in pairs]
print("using", len(pairs), "pairs")
print("example target:", ur_texts[0])

using 8000 pairs
example target: <sos> زین تمہارا بھتیجا ہے۔ <eos>


### Tokenize both languages
Each language gets its **own** tokenizer (separate vocabularies). Note `filters=''` for Urdu so the `<sos>`/`<eos>` tokens aren't stripped.

In [5]:
en_tok = Tokenizer()
en_tok.fit_on_texts(en_texts)
en_vocab = len(en_tok.word_index) + 1

ur_tok = Tokenizer(filters='')          # keep <sos> and <eos>
ur_tok.fit_on_texts(ur_texts)
ur_vocab = len(ur_tok.word_index) + 1

print("English vocab:", en_vocab, "| Urdu vocab:", ur_vocab)

English vocab: 3244 | Urdu vocab: 3466


In [6]:
# convert to padded integer sequences
en_seq = en_tok.texts_to_sequences(en_texts)
ur_seq = ur_tok.texts_to_sequences(ur_texts)

MAX_EN = max(len(s) for s in en_seq)
MAX_UR = max(len(s) for s in ur_seq)
en_seq = pad_sequences(en_seq, maxlen=MAX_EN, padding="post")
ur_seq = pad_sequences(ur_seq, maxlen=MAX_UR, padding="post")
print("English padded:", en_seq.shape, "| Urdu padded:", ur_seq.shape)

English padded: (8000, 7) | Urdu padded: (8000, 10)


### Teacher forcing: set up decoder input & target

**Teacher forcing** means: during training we feed the decoder the **correct previous Urdu word** (not its own guess), so it learns fast and stably.

- **decoder input** = the Urdu sequence **without the last word** (starts with `<sos>`)
- **decoder target** = the Urdu sequence **shifted one step left** (what it should predict next)

```
 input : <sos>  میں   طالب  علم   ہوں
 target:  میں   طالب  علم   ہوں   <eos>
```

In [7]:
decoder_input  = ur_seq[:, :-1]   # everything except the last token
decoder_target = ur_seq[:, 1:]    # shifted left by one
print("decoder input :", decoder_input.shape)
print("decoder target:", decoder_target.shape)

decoder input : (8000, 9)
decoder target: (8000, 9)


---
## 3. Build the seq2seq model (learned embeddings)

We build it directly — encoder on top, decoder below — so you can read it top to bottom:
- the **encoder** turns the English sentence into one context state,
- the **decoder** starts from that state and predicts the Urdu words.

We keep the layers as named variables (`decoder_gru`, `decoder_dense`, …) because we reuse them for translation later.

In [8]:
LATENT = 256      # size of the GRU hidden state (the 'memory')
EMB_DIM = 100     # embedding dimension

# ---------- ENCODER: English sentence -> context state ----------
encoder_inputs = Input(shape=(MAX_EN,))
encoder_emb_obj = Embedding(en_vocab, EMB_DIM)
encoder_emb = encoder_emb_obj(encoder_inputs)
_, context_state = GRU(LATENT, return_state=True)(encoder_emb)

# ---------- DECODER: starts from context, predicts Urdu words ----------
decoder_inputs = Input(shape=(MAX_UR - 1,))
decoder_embedding = Embedding(ur_vocab, EMB_DIM)          # named: reused at inference
decoder_gru = GRU(LATENT, return_sequences=True, return_state=True)
decoder_dense = Dense(ur_vocab, activation='softmax')

dec_emb = decoder_embedding(decoder_inputs)
dec_seq, _ = decoder_gru(dec_emb, initial_state=context_state)   # begin from encoder's context
decoder_outputs = decoder_dense(dec_seq)

# the full training model: (English, Urdu-so-far) -> next Urdu words
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 7)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, 9)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 7, 100)    │    324,400 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 9, 100)    │    346,600 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gru (GRU)           │ [(None, 256),     │    274,944 │ embedding[0][0]   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gru_1 (GRU)         │ [(None, 9, 256),  │    274,944 │ embedding_1[0][0… │
│                     │ (None, 256)]      │            │ gru[0][1]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 9, 3466)   │    890,762 │ gru_1[0][0]       │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,111,650 (8.06 MB)

 Trainable params: 2,111,650 (8.06 MB)

 Non-trainable params: 0 (0.00 B)

In [9]:
# quick check of the shapes going in
print('encoder input :', en_seq.shape)
print('decoder input :', decoder_input.shape)
print('decoder target:', decoder_target.shape)

encoder input : (8000, 7)
decoder input : (8000, 9)
decoder target: (8000, 9)


In [10]:
history = model.fit(
    [en_seq, decoder_input], decoder_target[..., None],
    epochs=30, batch_size=32, validation_split=0.1, verbose=1)

Epoch 1/30
225/225 ━━━━━━━━━━━━━━━━━━━━ 25s 101ms/step - accuracy: 0.3841 - loss: 4.7023 - val_accuracy: 0.4887 - val_loss: 3.0830
Epoch 2/30
225/225 ━━━━━━━━━━━━━━━━━━━━ 21s 95ms/step - accuracy: 0.5093 - loss: 2.9556 - val_accuracy: 0.5383 - val_loss: 2.7976
Epoch 3/30
225/225 ━━━━━━━━━━━━━━━━━━━━ 27s 120ms/step - accuracy: 0.5496 - loss: 2.6349 - val_accuracy: 0.5721 - val_loss: 2.6095
Epoch 4/30
225/225 ━━━━━━━━━━━━━━━━━━━━ 34s 152ms/step - accuracy: 0.5873 - loss: 2.3498 - val_accuracy: 0.5954 - val_loss: 2.4714
Epoch 5/30
 72/225 ━━━━━━━━━━━━━━━━━━━━ 20s 135ms/step - accuracy: 0.6049 - loss: 2.1521

: 

---
## 4. Translate new sentences (inference)

At translation time we don't have the correct Urdu words to feed in, so we generate one word at a time. We reuse the **same trained layers** to make two small models:
- **encoder_model**: English sentence → context state,
- **decoder_step**: (one Urdu word + state) → (next word + new state).

In [ ]:
# encoder for inference: reuse the encoder we already built
encoder_model = Model(encoder_inputs, context_state)

# step decoder: feed ONE word + the previous state, get the next word + new state
step_word = Input(shape=(1,))
step_state_in = Input(shape=(LATENT,))

step_emb = decoder_embedding(step_word)                       # same embedding as training
step_seq, step_state_out = decoder_gru(step_emb, initial_state=step_state_in)  # same GRU
step_probs = decoder_dense(step_seq)                          # same dense
decoder_step = Model([step_word, step_state_in], [step_probs, step_state_out])

idx_to_word = {i: w for w, i in ur_tok.word_index.items()}
print('inference models ready')

In [ ]:
def translate(sentence):
    # 1. encode the English sentence into a context state
    seq = pad_sequences(en_tok.texts_to_sequences([sentence.lower()]), maxlen=MAX_EN, padding='post')
    state = encoder_model.predict(seq, verbose=0)

    # 2. start the decoder with <sos>, then generate word by word
    word = np.array([[ur_tok.word_index['<sos>']]])
    output = []
    for _ in range(MAX_UR):
        probs, state = decoder_step.predict([word, state], verbose=0)
        idx = int(probs[0, -1].argmax())
        w = idx_to_word.get(idx, '')
        if w == '<eos>' or w == '':
            break
        output.append(w)
        word = np.array([[idx]])          # feed this prediction back in
    return ' '.join(output)

for s in ['i am happy', 'how are you', 'what is your name', 'i am a student']:
    print(f'{s:22s} -> {translate(s)}')

The translations won't be perfect on a small dataset trained briefly — but you should see it produce real Urdu words in a sensible order. **You built a translator.**

---
## 5. Swap the embedding: learned → Word2Vec → GloVe

So far the **English** embeddings were learned from scratch on our small dataset. But we can instead start from **pretrained** embeddings that already know word meanings from huge corpora — the same **transfer-learning** idea as vision.

We'll try two pretrained sources for the **English** side:
- **GloVe** (Stanford) and **Word2Vec** (Google) — both give a vector per English word.

*(Urdu stays learned-from-scratch: good pretrained Urdu vectors are less standard, so this cleanly isolates the effect of pretrained English embeddings.)*

In [ ]:
import gensim.downloader as api

# download pretrained vectors (needs internet; cached after first time)
print("downloading GloVe (100-dim)...")
glove = api.load("glove-wiki-gigaword-100")     # 100-dim to match EMB_DIM
print("downloading Word2Vec (300-dim)...")
w2v = api.load("word2vec-google-news-300")
print("done. glove dim:", glove.vector_size, "| w2v dim:", w2v.vector_size)

### Build an embedding matrix from pretrained vectors
For each English word in **our** vocabulary, look up its pretrained vector and place it in a matrix. Words not found keep a zero row. We then load this matrix into a (frozen) `Embedding` layer.

In [ ]:
def make_pretrained_embedding(kv, dim):
    """Build a Keras Embedding for the ENGLISH vocab from pretrained vectors kv."""
    matrix = np.zeros((en_vocab, dim))
    found = 0
    for word, i in en_tok.word_index.items():
        if word in kv:
            matrix[i] = kv[word]
            found += 1
    print(f"  matched {found}/{en_vocab-1} English words in the pretrained vectors")
    return Embedding(en_vocab, dim, weights=[matrix], trainable=False, name="en_emb_pretrained")

In [ ]:
def train_with_embedding(en_embedding, dim, label):
    """Build a fresh seq2seq using the given ENGLISH embedding, train it, return val accuracy."""
    print(f'\n=== {label} ===')
    # encoder
    e_in = Input(shape=(MAX_EN,))
    e = en_embedding(e_in)
    _, ctx = GRU(LATENT, return_state=True)(e)
    # decoder (urdu embedding always learned, matching the chosen dim)
    d_in = Input(shape=(MAX_UR - 1,))
    d = Embedding(ur_vocab, dim)(d_in)
    dseq, _ = GRU(LATENT, return_sequences=True, return_state=True)(d, initial_state=ctx)
    out = Dense(ur_vocab, activation='softmax')(dseq)
    m = Model([e_in, d_in], out)
    m.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    h = m.fit([en_seq, decoder_input], decoder_target[..., None],
              epochs=30, batch_size=128, validation_split=0.1, verbose=0)
    acc = h.history['val_accuracy'][-1]
    print(f'{label}: validation accuracy {acc:.2%}')
    return acc

results = {}
# 1. learned from scratch (100-dim, for a fair comparison)
results['Learned'] = train_with_embedding(Embedding(en_vocab, 100), 100, 'Learned (from scratch)')
# 2. GloVe (100-dim)
results['GloVe'] = train_with_embedding(make_pretrained_embedding(glove, 100), 100, 'GloVe (pretrained)')
# 3. Word2Vec (300-dim)
results['Word2Vec'] = train_with_embedding(make_pretrained_embedding(w2v, 300), 300, 'Word2Vec (pretrained)')

In [ ]:
import matplotlib.pyplot as plt

names = list(results.keys())
accs = [results[n] * 100 for n in names]
plt.bar(names, accs, color=["gray", "steelblue", "green"])
plt.ylabel("validation accuracy (%)")
plt.title("English embedding source: learned vs GloVe vs Word2Vec")
for i, v in enumerate(accs):
    plt.text(i, v + 0.3, f"{v:.1f}", ha="center")
plt.show()

print("summary:")
for n in names:
    print(f"  {n:10s} {results[n]:.2%}")

## Save the model and dictionaries

In [ ]:
# ============================================================
# SAVE CELL — run this AFTER training the model in the Day 5 notebook
# (i.e. after Part 4, once `encoder_model` and `decoder_step` exist).
# It writes a `translator/` folder that the Streamlit app (app.py) loads.
# ============================================================
import os
import pickle

os.makedirs("translator", exist_ok=True)

# 1. save the two inference models (NOT the training model — the app needs these)
encoder_model.save("translator/encoder.keras")
decoder_step.save("translator/decoder_step.keras")

# 2. save the tokenizers + the sizes the app needs to prepare and decode text
with open("translator/config.pkl", "wb") as f:
    pickle.dump(
        {
            "en_tok": en_tok,
            "ur_tok": ur_tok,
            "MAX_EN": MAX_EN,
            "MAX_UR": MAX_UR,
            "LATENT": LATENT,
        },
        f,
    )

print("saved to translator/:")
for fn in sorted(os.listdir("translator")):
    kb = os.path.getsize(f"translator/{fn}") / 1024
    print(f"  translator/{fn}  ({kb:.0f} KB)")
print("\nDownload the whole `translator/` folder, put app.py next to it, then:")
print("  pip install streamlit tensorflow")
print("  streamlit run app.py")

**What to look for:**
- Pretrained embeddings (**GloVe/Word2Vec**) often help most when the training data is **small** — they bring in word meaning our tiny corpus can't teach.
- On a very small/clean dataset the learned embeddings can catch up, since the model can memorize. The benefit of pretrained vectors grows as vocabulary and rarity increase.
- This is the **same transfer-learning lesson** as vision: *reuse* knowledge from a big corpus instead of learning everything from scratch. *(Exact numbers vary per run.)*

---
## Your turn (solo task) ✍️

Pick at least two:
1. **Translate your own sentences** with the trained model — which ones work, which fail, and why?
2. **Train longer** (60 epochs) or on **more pairs** (raise `MAX_PAIRS`) — do translations improve?
3. **Compare embeddings fairly** — make GloVe and the learned version both 100-dim (already done) and read the gap.
4. **Make the decoder deeper** or raise `LATENT` — effect on translation quality?
5. **Think ahead:** the encoder crams the *whole* sentence into one context vector. What happens for a *long* sentence? (This is the problem **attention** solves — next.)

In [ ]:
# ===== YOUR EXPERIMENTS HERE =====



---
## Summary

- **Seq2seq translation** uses **two GRUs**: an **encoder** that compresses the English sentence into a context vector, and a **decoder** that generates Urdu word-by-word from it.
- **`<sos>`/`<eos>`** tokens mark where the decoder starts and stops.
- **Teacher forcing** feeds the correct previous word during training for fast, stable learning; at **inference** we feed the model's own predictions back, one word at a time.
- **Embeddings:** we trained with **learned**, then **GloVe**, then **Word2Vec** English embeddings — pretrained vectors are transfer learning for language, most helpful when data is limited.

### The limitation to remember
The encoder squeezes the **entire** sentence into **one** vector. For long sentences, that's a bottleneck — the context vector "forgets" the start.

> **That bottleneck is exactly what *attention* fixes** — letting the decoder look back at every input word. That's where we go next: **attention, then Transformers.**

**This completes Week 11's sequence-models arc:** RNN → LSTM/GRU → embeddings → seq2seq translation → (next) attention.